# Introduction to Neural Networks

Classical machine learning algorithms — linear regression, decision trees, random forests — work well on structured tabular data with meaningful features. But images, speech, and language contain patterns too complex for hand-crafted features. **Neural networks** learn representations directly from raw data by stacking layers of simple computational units called neurons.

A single neuron is a weighted sum plus a nonlinear activation — not much more than logistic regression. The breakthrough comes from connecting thousands or millions of neurons in layers, allowing the network to learn hierarchical features: edges in early layers, shapes in middle layers, objects in deeper layers.

This notebook introduces the building blocks of neural networks: the artificial neuron, the perceptron, multi-layer architectures, activation functions, forward propagation, and training fundamentals. These concepts form the foundation for deep learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. From Neurons to Networks

The brain contains billions of **neurons** connected in vast networks. Each neuron receives electrical signals from other neurons, processes them, and fires its own signal if the combined input exceeds a threshold. Learning happens by strengthening or weakening connections between neurons.

An **artificial neuron** mimics this idea mathematically:

1. Receive inputs $x_1, x_2, \ldots, x_d$ (features).
2. Multiply each input by a **weight** $w_i$ and sum them, plus a **bias** $b$.
3. Pass the result through an **activation function** to produce an output.

$$z = w_1 x_1 + w_2 x_2 + \cdots + w_d x_d + b = \mathbf{w}^T \mathbf{x} + b$$

$$\hat{y} = f(z)$$

A single neuron is essentially a linear model followed by a nonlinear activation. The power comes from **connecting many neurons in layers** — each layer learns increasingly abstract representations of the input.

---

## 2. The Perceptron

The perceptron, invented by Frank Rosenblatt in 1958, is the simplest neural network — a single neuron with a step activation function:

$$\hat{y} = \begin{cases} 1 & \text{if } \mathbf{w}^T \mathbf{x} + b \geq 0 \\ 0 & \text{otherwise} \end{cases}$$

The perceptron learns by updating weights when it makes a mistake:

$$w_i \leftarrow w_i + \eta (y - \hat{y}) x_i$$

where $\eta$ is the learning rate and $y$ is the true label.

A single perceptron can only learn **linearly separable** patterns — it can draw a straight line (or hyperplane) between classes. It cannot solve XOR or other nonlinear problems. This limitation was famously highlighted by Minsky and Papert in 1969, contributing to the first "AI winter." The solution: stack multiple layers of neurons.

In [ ]:
# Perceptron on linearly separable vs non-separable data
X_sep, y_sep = make_moons(n_samples=200, noise=0.15, random_state=0)
X_lin, y_lin = make_moons(n_samples=200, noise=0.0, random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, X, y, title in zip(axes,
    [X_lin, X_sep],
    [y_lin, y_sep],
    ["Low noise (nearly separable)", "Higher noise (non-separable)"]):
    perc = Perceptron(max_iter=1000).fit(X, y)
    acc = accuracy_score(y, perc.predict(X))

    xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 150),
                          np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 150))
    Z = perc.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=20)
    ax.set_title(f"Single Perceptron\n{title}, acc={acc:.3f}")

plt.tight_layout()
plt.show()

---

## 3. Multi-Layer Perceptrons (MLP)

A **multi-layer perceptron** stacks neurons in layers:

```
  Input Layer    Hidden Layer(s)    Output Layer
  ┌───┐ ┌───┐      ┌───┐ ┌───┐         ┌───┐
  │ x₁│ │ x₂│ ──→  │ h₁│ │ h₂│ ──→     │ ŷ │
  └───┘ └───┘      └───┘ └───┘         └───┘
  (features)      (learned reps)      (prediction)
```

- **Input layer** — receives raw features (no computation).
- **Hidden layers** — learn intermediate representations. Each neuron applies weights, bias, and activation.
- **Output layer** — produces the final prediction (class probabilities or numeric value).

Each layer transforms the previous layer's output:

$$\mathbf{h}^{(l)} = f\left(\mathbf{W}^{(l)} \mathbf{h}^{(l-1)} + \mathbf{b}^{(l)}\right)$$

With enough hidden neurons and nonlinear activations, an MLP can approximate **any continuous function** — this is the **universal approximation theorem**. In practice, depth (more layers) often learns more efficiently than width (more neurons per layer).

---

## 4. Activation Functions

Activation functions introduce **nonlinearity**. Without them, stacking layers would be equivalent to a single linear transformation — no matter how many layers, the network could only learn linear patterns.

| Function | Formula | Range | Use Case |
|----------|---------|-------|----------|
| **Sigmoid** | $\sigma(z) = \frac{1}{1+e^{-z}}$ | (0, 1) | Binary output, historical hidden layers |
| **Tanh** | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | (-1, 1) | Hidden layers (zero-centered) |
| **ReLU** | $\text{ReLU}(z) = \max(0, z)$ | [0, ∞) | Default for hidden layers |
| **Softmax** | $\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | (0, 1), sums to 1 | Multi-class output |

**ReLU** (Rectified Linear Unit) is the modern default. It is simple, fast, and avoids the **vanishing gradient problem** that plagued sigmoid and tanh in deep networks. Variants like Leaky ReLU and GELU address edge cases where ReLU "dies" (neurons that always output zero).

In [ ]:
# Activation functions
z = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))

axes[0].plot(z, 1 / (1 + np.exp(-z)), "b-", linewidth=2)
axes[0].set_title("Sigmoid")
axes[0].axhline(0, color="gray", linewidth=0.5)
axes[0].axvline(0, color="gray", linewidth=0.5)

axes[1].plot(z, np.tanh(z), "g-", linewidth=2)
axes[1].set_title("Tanh")
axes[1].axhline(0, color="gray", linewidth=0.5)
axes[1].axvline(0, color="gray", linewidth=0.5)

axes[2].plot(z, np.maximum(0, z), "r-", linewidth=2)
axes[2].set_title("ReLU")
axes[2].axhline(0, color="gray", linewidth=0.5)
axes[2].axvline(0, color="gray", linewidth=0.5)

z3 = np.array([1.0, 2.0, 0.5])
softmax = np.exp(z3) / np.exp(z3).sum()
axes[3].bar(["z₁", "z₂", "z₃"], softmax, color="purple", alpha=0.7)
axes[3].set_title("Softmax (example)")
axes[3].set_ylim(0, 1)

for ax in axes[:3]:
    ax.set_xlabel("z")

plt.tight_layout()
plt.show()

---

## 5. Forward Propagation

**Forward propagation** computes the network's output by passing input through each layer sequentially:

1. Input $\mathbf{x}$ enters the first hidden layer.
2. Compute $\mathbf{z}^{(1)} = \mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)}$, then $\mathbf{h}^{(1)} = f(\mathbf{z}^{(1)})$.
3. Pass $\mathbf{h}^{(1)}$ to the next layer, repeat.
4. Final layer produces output $\hat{y}$.

For a network with one hidden layer of 2 neurons and one output neuron:

$$h_1 = \text{ReLU}(w_{11} x_1 + w_{12} x_2 + b_1)$$

$$h_2 = \text{ReLU}(w_{21} x_1 + w_{22} x_2 + b_2)$$

$$\hat{y} = \sigma(v_1 h_1 + v_2 h_2 + c)$$

During **training**, the network compares $\hat{y}$ to the true label $y$, computes a **loss**, and uses **backpropagation** to calculate how each weight contributed to the error. Weights are then updated via **gradient descent** to reduce the loss. This cycle repeats for many epochs until the network converges.

In [ ]:
# Manual forward propagation for a tiny network
def relu(z):
    return np.maximum(0, z)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x = np.array([0.5, -1.0])
W1 = np.array([[0.8, -0.3], [0.2, 0.6]])
b1 = np.array([0.1, -0.2])
v = np.array([0.5, -0.4])
c = 0.0

z1 = W1 @ x + b1
h = relu(z1)
y_hat = sigmoid(v @ h + c)

print(f"Input x:         {x}")
print(f"Hidden z:        {z1}")
print(f"Hidden h (ReLU): {h}")
print(f"Output ŷ:        {y_hat:.4f}")

---

## 6. Training Neural Networks

Training minimizes a **loss function** using **gradient descent**:

$$w \leftarrow w - \eta \frac{\partial L}{\partial w}$$

where $\eta$ is the learning rate and $L$ is the loss (cross-entropy for classification, MSE for regression).

**Backpropagation** efficiently computes gradients by applying the chain rule layer by layer, from output back to input. It is the algorithm that makes training deep networks practical.

Key training concepts:

- **Epoch** — one complete pass through the training data.
- **Batch size** — number of samples processed before updating weights.
- **Learning rate** — step size for weight updates.
- **Optimizer** — SGD, Adam, RMSprop — algorithms that adapt learning or use momentum.

**Regularization for neural networks:**
- **L2 weight decay** — penalize large weights.
- **Dropout** — randomly disable neurons during training.
- **Early stopping** — halt when validation loss stops improving.
- **Batch normalization** — normalize layer inputs for stable training.

In [ ]:
# MLP vs traditional models on moon-shaped data
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP (1 hidden, 10)": MLPClassifier(hidden_layer_sizes=(10,), max_iter=1000, random_state=42),
    "MLP (2 hidden, 20-10)": MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=1000, random_state=42),
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
xx, yy = np.meshgrid(np.linspace(X[:, 0].min()-0.5, X[:, 0].max()+0.5, 150),
                      np.linspace(X[:, 1].min()-0.5, X[:, 1].max()+0.5, 150))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_tr_s, y_tr)
    acc = accuracy_score(y_te, model.predict(X_te_s))
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr, cmap="coolwarm", edgecolors="k", s=10)
    ax.set_title(f"{name}\nacc={acc:.3f}", fontsize=9)

plt.tight_layout()
plt.show()

---

## 7. When to Use Neural Networks

Neural networks are not always the best choice. Guidelines:

**Neural networks excel when:**
- Data is unstructured — images, text, audio, video.
- Large amounts of training data are available.
- Patterns are highly nonlinear and hierarchical.
- Feature engineering is difficult or unknown.

**Traditional ML may be better when:**
- Data is tabular with meaningful features.
- Dataset is small (hundreds to low thousands of samples).
- Interpretability is critical.
- Fast training and inference are required.
- Limited computational resources.

For tabular data, gradient boosting often matches or beats neural networks with far less tuning. Neural networks become essential for images, language, and other high-dimensional unstructured data — the domain of **deep learning**, where networks with many layers learn increasingly abstract representations.

---

## 8. Summary

Neural networks extend linear models into powerful function approximators by stacking layers of neurons with nonlinear activations. The **perceptron** is the simplest unit; **multi-layer perceptrons** combine many neurons to learn complex patterns. **Activation functions** introduce nonlinearity; **forward propagation** computes predictions; **backpropagation** and **gradient descent** learn weights from data.

When data is unstructured and abundant, neural networks outperform hand-engineered features. Adding depth — many layers — creates **deep learning**, capable of learning hierarchical representations that power modern computer vision, speech recognition, and language models.